 ## Silver — conform the clinical and financial entities.

 Bronze holds everything as strings, exactly as it arrived. This notebook
 turns that into typed, deduplicated, standardized tables that Gold can
 build dimensions and facts from.

 Four things happen to every entity:
   1. Cast to real types, quarantining rows that fail
   2. Standardize coded values through ref_code_mapping
   3. Deduplicate on the business key using _row_hash
   4. Resolve patient identity through silver_patient_xref

 Nothing is silently dropped. A row that fails casting goes to a quarantine
 table with the reason attached, so row counts always reconcile to Bronze
 and the failures are visible in the data-quality report rather than
 discovered when a total looks wrong six weeks later.

In [ ]:
# batch_id = "SILVER_MANUAL"

In [ ]:
BRONZE = "lh_bronze.dbo"

In [ ]:

from datetime import datetime, timezone

from pyspark.sql import functions as F, Window

run_ts = datetime.now(timezone.utc)
quarantine_summary = []
load_summary = []

print(f"Silver conformance run {batch_id}")

### Helpers

In [ ]:
def code_map(domain, source_system):
    """One row per source code for a domain, ready to join."""
    return (spark.table(f"{BRONZE}.ref_code_mapping")
            .filter((F.col("domain") == domain)
                    & (F.col("source_system") == source_system)
                    & (F.lower(F.col("is_active")) == "true"))
            .select(F.upper(F.trim(F.col("source_code"))).alias("_src"),
                    F.col("standard_code").alias("_std")))


def apply_mapping(df, col_name, domain, source_system, out_col=None):
    """Map a coded column to its standard code, defaulting to UNKNOWN.

    Unmapped codes are not dropped and not silently defaulted in silence —
    they are counted below and written to silver_unmapped_codes so a steward
    can extend the mapping without a code change or a reload.
    """
    out_col = out_col or col_name
    m = code_map(domain, source_system)
    joined = df.join(m, F.upper(F.trim(F.col(col_name))) == F.col("_src"), "left")
    return joined.withColumn(out_col,
                             F.coalesce(F.col("_std"), F.lit("UNKNOWN"))).drop("_src", "_std")


def record_unmapped(df, col_name, domain, source_system):
    unmapped = (df.filter(F.col(col_name) == "UNKNOWN")
                  .select(F.lit(domain).alias("domain"),
                          F.lit(source_system).alias("source_system"),
                          F.lit(col_name).alias("column_name"))
                  .groupBy("domain", "source_system", "column_name").count())
    if unmapped.count():
        (unmapped.withColumn("batch_id", F.lit(batch_id))
                 .withColumn("logged_ts", F.lit(run_ts))
                 .write.format("delta").mode("append").option("mergeSchema", "true")
                 .saveAsTable("silver_unmapped_codes"))


def quarantine(df, condition, entity, reason):
    """Split a frame into (passing, failing). Failing rows are persisted.

    Returns only the passing rows. The caller carries on with clean data and
    the rejects remain inspectable and replayable.
    """
    bad = df.filter(~condition | condition.isNull())
    n_bad = bad.count()
    if n_bad:
        (bad.withColumn("_dq_reason", F.lit(reason))
            .withColumn("_dq_batch_id", F.lit(batch_id))
            .withColumn("_dq_ts", F.lit(run_ts))
            .write.format("delta").mode("append").option("mergeSchema", "true")
            .saveAsTable(f"silver_quarantine_{entity}"))
        quarantine_summary.append((entity, reason, n_bad))
    return df.filter(condition)


def dedupe(df, business_keys, order_col="_ingest_ts"):
    """Keep the most recent row per business key.

    Bronze is append-only, so an incrementally-loaded entity accumulates one
    row per change. Silver wants current state, which is the latest row by
    ingest time — with _row_hash breaking ties deterministically so two runs
    produce the same result.
    """
    w = Window.partitionBy(*business_keys).orderBy(
        F.col(order_col).desc(), F.col("_row_hash").desc())
    return (df.withColumn("_rn", F.row_number().over(w))
              .filter(F.col("_rn") == 1).drop("_rn"))


def finish(df, table, entity, source_rows):
    (df.withColumn("_silver_batch_id", F.lit(batch_id))
       .withColumn("_silver_ts", F.lit(run_ts))
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(table))
    n = spark.table(table).count()
    load_summary.append((entity, source_rows, n, source_rows - n))
    print(f"  {entity:<28} bronze={source_rows:>8,}  silver={n:>8,}")



## Patient cross-reference

 Every entity below resolves its patient through this, never through a
 source system's own id. That single rule is what makes cross-facility
 analysis possible — a readmission at a different hospital is only visible
 if both admissions resolve to the same golden id.

In [ ]:
xref = spark.table("silver_patient_xref").select("record_uid", "patient_golden_id")


def resolve_patient(df, source_system, id_col):
    return (df.withColumn("_uid", F.concat_ws("|", F.lit(source_system), F.col(id_col)))
              .join(xref, F.col("_uid") == F.col("record_uid"), "left")
              .drop("_uid", "record_uid"))

## 1. Admissions

In [ ]:
src = spark.table(f"{BRONZE}.ehr_admission")
n_src = src.count()

adm = dedupe(src, ["admission_id"])
adm = resolve_patient(adm, "EHR", "patient_id")

adm = (adm
    .withColumn("admission_ts", F.to_timestamp("admission_ts"))
    .withColumn("discharge_ts", F.to_timestamp("discharge_ts"))
    .withColumn("expected_discharge_ts", F.to_timestamp("expected_discharge_ts")))

# An admission with no timestamp cannot be placed on a calendar, so it
# cannot contribute to any measure. Quarantine rather than carry a null.
adm = quarantine(adm, F.col("admission_ts").isNotNull(),
                 "admission", "admission_ts is null or unparseable")

# A discharge before an admission is a source clock error. Null the
# discharge rather than drop the admission — the admission happened, and a
# negative length of stay would corrupt every average it touches.
adm = adm.withColumn("_clock_error",
                     F.col("discharge_ts").isNotNull()
                     & (F.col("discharge_ts") < F.col("admission_ts")))
n_clock = adm.filter("_clock_error").count()
if n_clock:
    print(f"  {n_clock} admissions with discharge before admission — "
          f"discharge nulled, flagged for DQ")
adm = adm.withColumn("discharge_ts",
                     F.when(F.col("_clock_error"), None).otherwise(F.col("discharge_ts")))

adm = apply_mapping(adm, "admission_type", "ADMIT_TYPE", "EHR", "admission_type_std")
adm = apply_mapping(adm, "discharge_disposition_code", "DISPOSITION", "EHR",
                    "discharge_disposition_std")
record_unmapped(adm, "admission_type_std", "ADMIT_TYPE", "EHR")

adm = (adm
    # Same-day admission and discharge counts as one day. That is the CIHI
    # convention and what clinicians expect to see.
    .withColumn("length_of_stay_days",
                F.when(F.col("discharge_ts").isNull(), None)
                 .otherwise(F.greatest(F.datediff("discharge_ts", "admission_ts"), F.lit(1))))
    .withColumn("length_of_stay_hours",
                F.when(F.col("discharge_ts").isNull(), None)
                 .otherwise((F.col("discharge_ts").cast("long")
                             - F.col("admission_ts").cast("long")) / 3600.0))
    .withColumn("is_open", F.col("discharge_ts").isNull()))

finish(adm.drop("_clock_error"), "silver_admission", "admission", n_src)

## 2. Emergency visits

In [ ]:
src = spark.table(f"{BRONZE}.ehr_emergency_visit")
n_src = src.count()

ed = dedupe(src, ["ed_visit_id"])
ed = resolve_patient(ed, "EHR", "patient_id")

for c in ["arrival_ts", "triage_ts", "physician_seen_ts",
          "admit_decision_ts", "bed_assigned_ts", "departure_ts"]:
    ed = ed.withColumn(c, F.to_timestamp(c))

ed = quarantine(ed, F.col("arrival_ts").isNotNull(),
                "emergency_visit", "arrival_ts is null or unparseable")

ed = ed.withColumn("ctas_level",
                   F.when(F.col("triage_score").cast("int").between(1, 5),
                          F.col("triage_score").cast("int")))
n_bad_ctas = ed.filter(F.col("ctas_level").isNull()).count()
if n_bad_ctas:
    print(f"  {n_bad_ctas} ED visits with CTAS outside 1-5 — nulled, flagged for DQ")


def minutes(a, b):
    """Interval in minutes, or null when the sequence is impossible.

    Negative intervals are source clock errors. They are nulled rather than
    clamped to zero: clamping would silently pull every wait-time average
    downward, which is exactly the metric these visits feed.
    """
    d = (F.col(b).cast("long") - F.col(a).cast("long")) / 60.0
    return F.when(F.col(a).isNull() | F.col(b).isNull(), None) \
            .when(d < 0, None).otherwise(d)


ed = (ed
    .withColumn("triage_wait_min", minutes("arrival_ts", "triage_ts"))
    # Physician initial assessment — the metric reported to the ministry.
    .withColumn("physician_initial_assessment_min", minutes("arrival_ts", "physician_seen_ts"))
    .withColumn("decision_to_admit_min", minutes("physician_seen_ts", "admit_decision_ts"))
    .withColumn("boarding_min", minutes("admit_decision_ts", "bed_assigned_ts"))
    .withColumn("total_ed_los_min", minutes("arrival_ts", "departure_ts"))
    .withColumn("left_without_being_seen",
                F.col("physician_seen_ts").isNull() & F.col("departure_ts").isNotNull())
    .withColumn("resulted_in_admission", F.col("resulted_in_admission") == "1"))

finish(ed, "silver_emergency_visit", "emergency_visit", n_src)

## 3. Appointments


In [ ]:
src = spark.table(f"{BRONZE}.sched_appointment")
n_src = src.count()

appt = dedupe(src, ["appointment_id"])
appt = resolve_patient(appt, "SCHED", "patient_ref")
appt = (appt.withColumnRenamed("patient_golden_id", "_gid_sched")
            .transform(lambda d: resolve_patient(d, "EHR", "patient_ref"))
            .withColumn("patient_golden_id",
                        F.coalesce(F.col("_gid_sched"), F.col("patient_golden_id")))
            .drop("_gid_sched"))

for c in ["booking_ts", "scheduled_ts", "checkin_ts", "seen_ts",
          "checkout_ts", "cancellation_ts"]:
    appt = appt.withColumn(c, F.to_timestamp(c))

appt = quarantine(appt, F.col("scheduled_ts").isNotNull(),
                  "appointment", "scheduled_ts is null or unparseable")

appt = (appt
    .withColumn("is_completed", F.col("status_code") == "COMPLETED")
    .withColumn("is_no_show", F.col("status_code") == "NO_SHOW")
    .withColumn("is_cancelled", F.col("status_code").startswith("CANCELLED"))
    .withColumn("lead_time_days", F.datediff("scheduled_ts", "booking_ts"))
    .withColumn("cancellation_notice_hours",
                F.when(F.col("cancellation_ts").isNotNull(),
                       (F.col("scheduled_ts").cast("long")
                        - F.col("cancellation_ts").cast("long")) / 3600.0))
    .withColumn("wait_in_clinic_min",
                F.when(F.col("checkin_ts").isNotNull() & F.col("seen_ts").isNotNull(),
                       (F.col("seen_ts").cast("long")
                        - F.col("checkin_ts").cast("long")) / 60.0))
    .withColumn("actual_duration_min",
                F.when(F.col("seen_ts").isNotNull() & F.col("checkout_ts").isNotNull(),
                       (F.col("checkout_ts").cast("long")
                        - F.col("seen_ts").cast("long")) / 60.0))
    .withColumn("scheduled_duration_min", F.col("scheduled_duration_min").cast("int"))
    .withColumn("is_virtual", F.col("is_virtual") == "1")
    .withColumn("is_first_visit", F.col("is_first_visit") == "1"))

finish(appt, "silver_appointment", "appointment", n_src)

## 4. Lab results

 The encounter id is the join back to admissions. Lab results arrive from a
 different system than the ADT feed, so a result whose encounter does not
 resolve is an orphan — it happened, but it cannot be attributed. Counted
 rather than discarded.

In [ ]:
src = spark.table(f"{BRONZE}.lis_lab_result")
n_src = src.count()

lab = dedupe(src, ["placer_order_id", "loinc_code", "set_id"])
lab = resolve_patient(lab, "EHR", "patient_id")

lab = (lab
    .withColumn("order_ts", F.col("order_ts").cast("timestamp"))
    .withColumn("collect_ts", F.col("collect_ts").cast("timestamp"))
    .withColumn("result_ts", F.col("result_ts").cast("timestamp"))
    .withColumn("result_value_numeric", F.col("result_value_numeric").cast("decimal(18,4)"))
    .withColumn("is_abnormal", F.col("abnormal_flag").isin("H", "L", "HH", "LL", "A", "AA"))
    .withColumn("is_critical", F.col("abnormal_flag").isin("HH", "LL", "AA")))

lab = lab.withColumn(
    "total_turnaround_min",
    F.when(F.col("order_ts").isNull() | F.col("result_ts").isNull(), None)
     .otherwise((F.col("result_ts").cast("long") - F.col("order_ts").cast("long")) / 60.0))

# A negative turnaround means the result predates the order. Quarantine —
# unlike a wait time, this cannot be salvaged by nulling one field, because
# which of the two timestamps is wrong is unknowable.
lab = quarantine(lab, F.col("total_turnaround_min").isNull()
                      | (F.col("total_turnaround_min") >= 0),
                 "lab_result", "result_ts before order_ts")

lab = lab.withColumn("loinc_code",
                     F.when(F.col("loinc_code").isin("", "LOCAL-99", "XX-000", "PENDING"),
                            None).otherwise(F.col("loinc_code")))

orphans = lab.filter(F.col("encounter_id").isNull() | (F.col("encounter_id") == "")).count()
if orphans:
    print(f"  {orphans:,} lab results with no encounter id — retained, unattributable")

finish(lab, "silver_lab_result", "lab_result", n_src)

 ## 5. Medication orders


In [ ]:
src = spark.table(f"{BRONZE}.pharm_medication_order")
n_src = src.count()

med = dedupe(src, ["medication_order_id"])
med = resolve_patient(med, "EHR", "patient_id")

med = (med
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("dispense_ts", F.to_timestamp("dispense_ts"))
    .withColumn("quantity_ordered", F.col("quantity_ordered").cast("decimal(12,2)"))
    .withColumn("quantity_dispensed", F.col("quantity_dispensed").cast("decimal(12,2)"))
    .withColumn("unit_cost", F.col("unit_cost").cast("decimal(12,4)"))
    .withColumn("total_cost", F.col("total_cost").cast("decimal(18,2)"))
    .withColumn("days_supply", F.col("days_supply").cast("int"))
    .withColumn("din", F.when(F.col("din").isin("", "00000000", "LOCAL01"), None)
                        .otherwise(F.col("din")))
    .withColumn("order_to_dispense_min",
                F.when(F.col("order_ts").isNull() | F.col("dispense_ts").isNull(), None)
                 .otherwise((F.col("dispense_ts").cast("long")
                             - F.col("order_ts").cast("long")) / 60.0)))

finish(med, "silver_medication_order", "medication_order", n_src)

## 6. Claims

 The 837 submission and the 835 remittance are separate feeds describing the
 same claim at different points in its life. Joining them here is what makes
 approval rate and days-to-adjudicate answerable.


In [ ]:
sub = spark.table(f"{BRONZE}.claims_837_service_line")
rem = spark.table(f"{BRONZE}.claims_835_remittance")
n_src = sub.count()

claim_hdr = (sub.groupBy("claim_number")
    .agg(F.first("payer_id", ignorenulls=True).alias("payer_id"),
         F.first("subscriber_member_id", ignorenulls=True).alias("patient_source_id"),
         F.first("billing_provider_npi", ignorenulls=True).alias("hospital_id"),
         F.first("primary_diagnosis_code", ignorenulls=True).alias("primary_diagnosis_code"),
         F.first("service_date", ignorenulls=True).alias("service_date"),
         F.first("transaction_date", ignorenulls=True).alias("submission_date"),
         F.sum(F.col("line_amount").cast("decimal(18,2)")).alias("billed_amount"),
         F.count("*").alias("line_count")))

remit = (rem.groupBy("claim_number")
    .agg(F.first("claim_status_code", ignorenulls=True).alias("status_code"),
         F.first("payment_date", ignorenulls=True).alias("payment_date"),
         F.sum(F.col("paid_amount").cast("decimal(18,2)")).alias("paid_amount"),
         F.sum(F.col("patient_responsibility").cast("decimal(18,2)")).alias("patient_responsibility"),
         F.sum(F.col("adjustment_amount").cast("decimal(18,2)")).alias("adjustment_amount"),
         F.first("adjustment_reason", ignorenulls=True).alias("denial_reason_code")))

claims = claim_hdr.join(remit, "claim_number", "left")

claims = (claims
    .withColumn("service_date", F.to_date("service_date", "yyyyMMdd"))
    .withColumn("submission_date", F.to_date("submission_date", "yyyyMMdd"))
    .withColumn("payment_date", F.to_date("payment_date", "yyyyMMdd"))
    # X12 835 claim status: 1 processed as primary, 2 as secondary,
    # 4 denied, 22 reversal.
    .withColumn("is_adjudicated", F.col("status_code").isNotNull())
    .withColumn("is_approved", F.col("status_code").isin("1", "2"))
    .withColumn("is_denied", F.col("status_code") == "4")
    .withColumn("denied_amount",
                F.when(F.col("status_code") == "4", F.col("billed_amount"))
                 .otherwise(F.coalesce(F.col("adjustment_amount"), F.lit(0))))
    .withColumn("days_to_payment", F.datediff("payment_date", "submission_date")))

finish(claims, "silver_claim", "claim", n_src)

## 7. Billing, surveys, and the remaining reference entities


In [ ]:
src = spark.table(f"{BRONZE}.fin_invoice_line")
n_src = src.count()
bill = dedupe(src, ["invoice_id", "invoice_line_number"])
bill = resolve_patient(bill, "EHR", "patient_id")
for c, t in [("quantity", "decimal(12,2)"), ("charge_amount", "decimal(18,2)"),
             ("discount_amount", "decimal(18,2)"), ("tax_amount", "decimal(18,2)"),
             ("net_amount", "decimal(18,2)"), ("payment_amount", "decimal(18,2)"),
             ("outstanding_amount", "decimal(18,2)")]:
    bill = bill.withColumn(c, F.col(c).cast(t))
finish(bill, "silver_billing_line", "billing_line", n_src)

src = spark.table(f"{BRONZE}.survey_response")
n_src = src.count()
srv = dedupe(src, ["survey_response_id"])
srv = resolve_patient(srv, "EHR", "patient_id")
for c in ["overall_score", "wait_time_score", "staff_courtesy_score",
          "cleanliness_score", "communication_score", "pain_management_score",
          "would_recommend_score"]:
    srv = srv.withColumn(c, F.col(c).cast("int"))
# Scores outside 1-10 are impossible; quarantine rather than let them skew
# an average that leadership reads monthly.
srv = quarantine(srv, F.col("overall_score").between(1, 10),
                 "survey_response", "overall_score outside 1-10")
srv = (srv.withColumn("response_date", F.to_date("response_date"))
          .withColumn("service_date", F.to_date("service_date")))
finish(srv, "silver_survey_response", "survey_response", n_src)

src = spark.table(f"{BRONZE}.ehr_diagnosis")
n_src = src.count()
dx = dedupe(src, ["encounter_id", "diagnosis_code", "diagnosis_rank"])
dx = (dx.withColumn("diagnosis_rank", F.col("diagnosis_rank").cast("int"))
        .withColumn("is_primary", F.col("is_primary") == "1")
        .withColumn("is_present_on_admission", F.col("is_present_on_admission") == "1")
        .withColumn("diagnosis_date", F.to_date("diagnosis_date")))
finish(dx, "silver_diagnosis", "diagnosis", n_src)

src = spark.table(f"{BRONZE}.ehr_bed_assignment")
n_src = src.count()
bed = dedupe(src, ["admission_id", "bed_id", "assignment_start_ts"])
bed = (bed.withColumn("assignment_start_ts", F.to_timestamp("assignment_start_ts"))
          .withColumn("assignment_end_ts", F.to_timestamp("assignment_end_ts")))
bed = quarantine(bed, F.col("assignment_start_ts").isNotNull(),
                 "bed_assignment", "assignment_start_ts is null")
finish(bed, "silver_bed_assignment", "bed_assignment", n_src)

for bronze, silver, keys in [
    ("facil_hospital", "silver_hospital", ["hospital_id"]),
    ("facil_department", "silver_department", ["department_id"]),
    ("facil_bed", "silver_bed", ["bed_id"]),
    ("hr_doctor", "silver_doctor", ["doctor_id"]),
]:
    s = spark.table(f"{BRONZE}.{bronze}")
    n = s.count()
    finish(dedupe(s, keys), silver, silver.replace("silver_", ""), n)

## Summary


In [1]:
print("\n" + "=" * 74)
print(f"{'entity':<28}{'bronze':>10}{'silver':>10}{'removed':>10}")
print("-" * 74)
for e, b, s_, d in load_summary:
    print(f"{e:<28}{b:>10,}{s_:>10,}{d:>10,}")
print("=" * 74)

if quarantine_summary:
    print("\nQuarantined:")
    for e, r, n in quarantine_summary:
        print(f"  {e:<24} {n:>7,}  {r}")
else:
    print("\nNothing quarantined.")

# Patient resolution rate is the check that matters most — an unresolved
# patient means the fact cannot join to dim_patient and will land on the
# unknown member in Gold.
print("\nPatient resolution:")
for t in ["silver_admission", "silver_emergency_visit", "silver_appointment",
          "silver_lab_result", "silver_medication_order", "silver_billing_line",
          "silver_survey_response"]:
    df = spark.table(t)
    n = df.count()
    unresolved = df.filter(F.col("patient_golden_id").isNull()).count()
    print(f"  {t:<28} {n:>9,}  unresolved={unresolved:>7,} "
          f"({100 * unresolved / max(1, n):.2f}%)")

import json
mssparkutils.notebook.exit(json.dumps({
    "batch_id": batch_id,
    "entities": len(load_summary),
    "quarantined": sum(n for _, _, n in quarantine_summary),
}))

StatementMeta(, a3759030-ce13-4fce-beec-b557e6a44d6c, 3, Finished, Available, Finished, False)

Silver conformance run SILVER_MANUAL
  58 admissions with discharge before admission — discharge nulled, flagged for DQ
  admission                    bronze=  11,568  silver=  11,568
  132 ED visits with CTAS outside 1-5 — nulled, flagged for DQ
  emergency_visit              bronze=  25,037  silver=  25,037
  appointment                  bronze= 228,833  silver= 228,833
  lab_result                   bronze= 161,728  silver= 161,319
  medication_order             bronze=  97,511  silver=  97,511
  claim                        bronze=   3,234  silver=     711
  billing_line                 bronze=  52,196  silver=  52,196
  survey_response              bronze=  16,998  silver=   8,471
  diagnosis                    bronze=  37,318  silver=  37,318
  bed_assignment               bronze=  14,804  silver=  14,804
  hospital                     bronze=       5  silver=       5
  department                   bronze=      49  silver=      49
  bed                          bronze=     101  s